# ScheduleManager: Creating Contribution Schedules

> **⚠️ Advanced Topic**: This notebook demonstrates direct usage of the
> `ScheduleManager` class, which is an **internal implementation detail** of the
> SinkingFund API. Most users should use the `SinkingFund` class instead,
> which provides a unified interface for all operations. This notebook is
> provided for advanced users who need to understand the internal
> architecture or extend the system.

The `ScheduleManager` coordinates the creation of contribution schedules for
envelopes using various scheduling strategies. It implements the Strategy pattern,
allowing different scheduling algorithms to be plugged in without changing the
core management logic.

**Topics covered:**
- Setting scheduling strategies
- Creating contribution schedules for envelopes
- Understanding ScheduleResult structure
- Configuring contribution intervals

**For most users**: Use `SinkingFund.schedule()` instead of creating
ScheduleManager directly.


## Import Required Modules


In [ ]:
from datetime import date
from decimal import Decimal
from sinkingfund.managers import ScheduleManager
from sinkingfund.models import BillInstance, Envelope


## Setting Up Envelopes

Before creating schedules, we need envelopes with contribution dates configured:


In [ ]:
# Create bill instances.
instances = [
    BillInstance(
        bill_id="property_tax",
        service="Property Tax 2025",
        due_date=date(2025, 11, 1),
        amount_due=Decimal("3600.00")
    ),
    BillInstance(
        bill_id="car_insurance",
        service="Car Insurance",
        due_date=date(2025, 6, 1),
        amount_due=Decimal("750.00")
    ),
]

# Create envelopes with contribution windows.
envelopes = []
for instance in instances:
    envelope = Envelope(
        bill_instance=instance,
        initial_allocation=Decimal("0.00"),
        start_contrib_date=date(2025, 1, 1),
        end_contrib_date=instance.due_date,
        contrib_interval=14  # Bi-weekly contributions
    )
    envelopes.append(envelope)

print(f"Created {len(envelopes)} envelopes:")
for envelope in envelopes:
    print(f"  {envelope.bill_instance.service}: ${envelope.bill_instance.amount_due} due {envelope.bill_instance.due_date}")


## Setting the Scheduling Strategy

ScheduleManager uses the Strategy pattern. First, we set the scheduler strategy:


In [ ]:
# Create ScheduleManager.
manager = ScheduleManager()

# Set the independent scheduler strategy (default).
manager.set_scheduler(strategy="independent_scheduler")

print("Scheduler strategy set: independent_scheduler")


## Creating Schedules

The `create_schedules()` method generates contribution schedules for all envelopes:


In [ ]:
# Create schedules for all envelopes.
result = manager.create_schedules(envelopes=envelopes)

print(f"Created schedules for {len(result.schedules)} envelopes")
print(f"Strategy used: {result.metadata.get('strategy', 'unknown')}")


## Understanding ScheduleResult

The `ScheduleResult` contains the generated schedules and metadata:


In [ ]:
# Access the schedules dictionary.
print("=== ScheduleResult Structure ===")
print(f"Schedules: {len(result.schedules)} envelopes")
print(f"Metadata: {result.metadata}")

# Each schedule maps an envelope to its CashFlowSchedule.
for envelope, schedule in result.schedules.items():
    print(f"\n{envelope.bill_instance.service}:")
    print(f"  Total cash flows: {len(schedule.cash_flows)}")
    if len(schedule.cash_flows) > 0:
        print(f"  First contribution: {schedule.cash_flows[0].date} - ${schedule.cash_flows[0].amount}")
        if len(schedule.cash_flows) > 1:
            print(f"  Last contribution: {schedule.cash_flows[-2].date} - ${schedule.cash_flows[-2].amount}")
            print(f"  Payment on due date: {schedule.cash_flows[-1].date} - ${schedule.cash_flows[-1].amount}")


## Applying Schedules to Envelopes

After creating schedules, you typically apply them to envelopes. In the SinkingFund API,
this is done automatically, but when using ScheduleManager directly, you need to assign them:


In [ ]:
# Apply schedules to envelopes.
for envelope, schedule in result.schedules.items():
    envelope.schedule = schedule

# Now envelopes have schedules attached.
print("Schedules applied to envelopes:")
for envelope in envelopes:
    if envelope.schedule:
        total_contributions = envelope.schedule.total_amount_as_of_date(
            envelope.bill_instance.due_date,
            exclude='payouts'
        )
        print(f"  {envelope.bill_instance.service}: ${total_contributions} in contributions")


## Independent Scheduler Behavior

The independent scheduler creates even contribution schedules for each bill
independently. Each envelope gets regular contributions based on its contribution interval:


In [ ]:
# Show contribution pattern for property tax envelope.
prop_tax_envelope = [e for e in envelopes if e.bill_instance.bill_id == "property_tax"][0]

print(f"=== {prop_tax_envelope.bill_instance.service} Schedule ===")
print(f"Target amount: ${prop_tax_envelope.bill_instance.amount_due}")
print(f"Contribution window: {prop_tax_envelope.start_contrib_date} to {prop_tax_envelope.end_contrib_date}")
print(f"Contribution interval: {prop_tax_envelope.contrib_interval} days (bi-weekly)")
print("\nFirst 5 contributions:")
contributions = [f for f in prop_tax_envelope.schedule.cash_flows if f.is_inflow]
for contrib in contributions[:5]:
    print(f"  {contrib.date}: ${contrib.amount}")


## Summary

**Key Points:**

1. **Strategy Selection**: Use `set_scheduler()` to choose a scheduling strategy (currently "independent_scheduler")
2. **Schedule Creation**: `create_schedules()` generates contribution schedules for all envelopes
3. **ScheduleResult**: Contains schedules dictionary and metadata about the operation
4. **Independent Scheduler**: Creates even, regular contributions for each bill independently
5. **Contribution Intervals**: Schedules respect the envelope's `contrib_interval` setting

**Next Steps:**
- See how schedules integrate with the complete SinkingFund workflow using `SinkingFund.schedule()`
- Learn about allocation strategies that distribute funds before scheduling
- Explore how envelopes use schedules to track balances over time
